# Practical Exam: House sales

RealAgents is a real estate company that focuses on selling houses.

RealAgents sells a variety of types of house in one metropolitan area.

Some houses sell slowly and sometimes require lowering the price in order to find a buyer.

In order to stay competitive, RealAgents would like to optimize the listing prices of the houses it is trying to sell.

They want to do this by predicting the sale price of a house given its characteristics.

If they can predict the sale price in advance, they can decrease the time to sale.


## Data

The dataset contains records of previous houses sold in the area.

| Column Name | Criteria                                                |
|-------------|---------------------------------------------------------|
| house_id    | Nominal. </br> Unique identifier for houses. </br>Missing values not possible. |
| city        | Nominal. </br>The city in which the house is located. One of 'Silvertown', 'Riverford', 'Teasdale' and 'Poppleton'. </br>Replace missing values with "Unknown". |
| sale_price  | Discrete. </br>The sale price of the house in whole dollars. Values can be any positive number greater than or equal to zero.</br>Remove missing entries. |
| sale_date   | Discrete. </br>The date of the last sale of the house. </br>Replace missing values with 2023-01-01. |
| months_listed  | Continuous. </br>The number of months the house was listed on the market prior to its last sale, rounded to one decimal place. </br>Replace missing values with mean number of months listed, to one decimal place. |
| bedrooms    | Discrete. </br>The number of bedrooms in the house. Any positive values greater than or equal to zero. </br>Replace missing values with the mean number of bedrooms, rounded to the nearest integer. |
| house_type   | Ordinal. </br>One of "Terraced" (two shared walls), "Semi-detached" (one shared wall), or "Detached" (no shared walls). </br>Replace missing values with the most common house type. |
| area      | Continuous. </br>The area of the house in square meters, rounded to one decimal place. </br>Replace missing values with the mean, to one decimal place. |


# Task 1

The team at RealAgents knows that the city that a property is located in makes a difference to the sale price. 

Unfortuntately they believe that this isn't always recorded in the data. 

Calculate the number of missing values of the `city`. 

 - You should use the data in the file "house_sales.csv". 

 - Your output should be an object `missing_city`, that contains the number of missing values in this column. 

In [7]:
# Use this cell to write your code for Task 1
missing_tokens <- c("", " ", "NA", "N/A", "n/a", "na", "NaN", "NULL", "null",
                    "-", "--", "?", "missing", "Missing", "MISSING",
                    "none", "None", "NONE")

house_sales <- read.csv("house_sales.csv", stringsAsFactors = FALSE,
                        na.strings = missing_tokens, strip.white = TRUE)

missing_city <- sum(is.na(house_sales$city))
missing_city      # should be 73

[1] 73

# Task 2 

Before you fit any models, you will need to make sure the data is clean. 

The table below shows what the data should look like. 

Create a cleaned version of the dataframe. 

 - You should start with the data in the file "house_sales.csv". 

 - Your output should be a dataframe named `clean_data`. 

 - All column names and values should match the table below.


| Column Name | Criteria                                                |
|-------------|---------------------------------------------------------|
| house_id    | Nominal. </br> Unique identifier for houses. </br>Missing values not possible. |
| city        | Nominal. </br>The city in which the house is located. One of 'Silvertown', 'Riverford', 'Teasdale' and 'Poppleton' </br>Replace missing values with "Unknown". |
| sale_price  | Discrete. </br>The sale price of the house in whole dollars. Values can be any positive number greater than or equal to zero.</br>Remove missing entries. |
| sale_date   | Discrete. </br>The date of the last sale of the house. </br>Replace missing values with 2023-01-01. |
| months_listed  | Continuous. </br>The number of months the house was listed on the market prior to its last sale, rounded to one decimal place. </br>Replace missing values with mean number of months listed, to one decimal place. |
| bedrooms    | Discrete. </br>The number of bedrooms in the house. Any positive values greater than or equal to zero. </br>Replace missing values with the mean number of bedrooms, rounded to the nearest integer. |
| house_type   | Ordinal. </br>One of "Terraced", "Semi-detached", or "Detached". </br>Replace missing values with the most common house type. |
| area      | Continuous. </br>The area of the house in square meters, rounded to one decimal place. </br>Replace missing values with the mean, to one decimal place. |

In [8]:
# Use this cell to write your code for Task 2
missing_tokens <- c("", " ", "NA", "N/A", "n/a", "na", "NaN", "NULL", "null",
                    "-", "--", "?", "missing", "Missing", "MISSING",
                    "none", "None", "NONE")

house_sales <- read.csv("house_sales.csv", stringsAsFactors = FALSE,
                        na.strings = missing_tokens, strip.white = TRUE)

clean_data <- house_sales

# Safe numeric conversion: keeps the leading number, drops units like "sq.m."
to_num <- function(x) {
  x <- trimws(as.character(x))
  ok <- grepl("^-?[0-9]*\\.?[0-9]+", x)
  x[!ok] <- NA
  x[ok] <- sub("^(-?[0-9]*\\.?[0-9]+).*$", "\\1", x[ok])
  as.numeric(x)
}

# ---- sale_price: convert, then remove missing entries ----
clean_data$sale_price <- to_num(clean_data$sale_price)
clean_data <- clean_data[!is.na(clean_data$sale_price), ]
clean_data$sale_price <- as.integer(round(clean_data$sale_price))

# ---- city: standardise text, replace missing/invalid with "Unknown" ----
valid_cities <- c("Silvertown", "Riverford", "Teasdale", "Poppleton")
clean_data$city <- tools::toTitleCase(tolower(trimws(clean_data$city)))
clean_data$city[!clean_data$city %in% valid_cities] <- "Unknown"

# ---- sale_date: parse, replace missing with 2023-01-01 ----
d <- trimws(as.character(clean_data$sale_date))
parsed <- as.Date(rep(NA_character_, length(d)))
for (f in c("%Y-%m-%d", "%Y/%m/%d", "%d/%m/%Y", "%d-%m-%Y")) {
  idx <- is.na(parsed) & !is.na(d)
  parsed[idx] <- as.Date(d[idx], format = f)
}
parsed[is.na(parsed)] <- as.Date("2023-01-01")
clean_data$sale_date <- parsed

# ---- months_listed: mean imputation, 1 decimal ----
clean_data$months_listed <- to_num(clean_data$months_listed)
clean_data$months_listed[is.na(clean_data$months_listed)] <-
  mean(clean_data$months_listed, na.rm = TRUE)
clean_data$months_listed <- round(clean_data$months_listed, 1)

# ---- bedrooms: mean imputation, nearest integer ----
clean_data$bedrooms <- to_num(clean_data$bedrooms)
clean_data$bedrooms[is.na(clean_data$bedrooms)] <-
  mean(clean_data$bedrooms, na.rm = TRUE)
clean_data$bedrooms <- as.integer(round(clean_data$bedrooms))

# ---- house_type: Det./Semi/Terr. -> full names, fill with most common ----
ht <- tolower(trimws(clean_data$house_type))
clean_data$house_type <- ifelse(grepl("^semi", ht), "Semi-detached",
                         ifelse(grepl("^det", ht),  "Detached",
                         ifelse(grepl("^terr", ht), "Terraced", NA)))
most_common <- names(which.max(table(clean_data$house_type)))
clean_data$house_type[is.na(clean_data$house_type)] <- most_common
clean_data$house_type <- factor(clean_data$house_type,
                                levels = c("Terraced", "Semi-detached", "Detached"),
                                ordered = TRUE)

# ---- area: remove "sq.m.", mean imputation, 1 decimal ----
clean_data$area <- to_num(clean_data$area)
clean_data$area[is.na(clean_data$area)] <- mean(clean_data$area, na.rm = TRUE)
clean_data$area <- round(clean_data$area, 1)

# ---- Checks ----
str(clean_data)
colSums(is.na(clean_data))     # all should be 0
table(clean_data$city)         # 4 cities + "Unknown" (73)
table(clean_data$house_type)   # only the 3 house types
summary(clean_data$area)       # numeric values, no NaN

'data.frame':	1500 obs. of  8 variables:
 $ house_id     : int  1217792 1900913 1174927 1773666 1258487 1539789 1353069 1155864 1608726 1358764 ...
 $ city         : chr  "Silvertown" "Silvertown" "Riverford" "Silvertown" ...
 $ sale_price   : int  55943 384677 281707 373251 328885 47143 429914 284440 87134 211203 ...
 $ sale_date    : Date, format: "2021-09-12" "2021-01-17" ...
 $ months_listed: num  5.4 6.3 6.9 6.1 8.7 5.1 5.9 5.8 4.4 3 ...
 $ bedrooms     : int  2 5 6 6 5 2 6 4 2 4 ...
 $ house_type   : Ord.factor w/ 3 levels "Terraced"<"Semi-detached"<..: 2 3 3 3 3 2 3 3 2 3 ...
 $ area         : num  108 499 542 528 477 ...


house_id          city    sale_price     sale_date months_listed 
            0             0             0             0             0 
     bedrooms    house_type          area 
            0             0             0


 Poppleton  Riverford Silvertown   Teasdale    Unknown 
       362        182        517        366         73 


     Terraced Semi-detached      Detached 
          131           245          1124 

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  100.3   228.4   350.7   352.1   478.0   600.0 

# Task 3 

The team at RealAgents have told you that they have always believed that the number of bedrooms is the biggest driver of house price. 

Producing a table showing the difference in the average sale price by number of bedrooms along with the variance to investigate this question for the team.

 - You should start with the data in the file 'house_sales.csv'.

 - Your output should be a data frame named `price_by_rooms`. 

 - It should include the three columns `bedrooms`, `avg_price`, `var_price`. 

 - Your answers should be rounded to 1 decimal place.   

In [19]:
missing_tokens <- c("", " ", "NA", "N/A", "n/a", "na", "NaN", "NULL", "null",
                    "-", "--", "?", "missing", "Missing", "MISSING",
                    "none", "None", "NONE")

house_sales <- read.csv("house_sales.csv", stringsAsFactors = FALSE,
                        na.strings = missing_tokens, strip.white = TRUE)

# Safe numeric conversion: keeps the leading number, drops units
to_num <- function(x) {
  x <- trimws(as.character(x))
  ok <- grepl("^-?[0-9]*\\.?[0-9]+", x)
  x[!ok] <- NA
  x[ok] <- sub("^(-?[0-9]*\\.?[0-9]+).*$", "\\1", x[ok])
  as.numeric(x)
}

df <- house_sales

# sale_price: remove missing entries (as per the data criteria)
df$sale_price <- to_num(df$sale_price)
df <- df[!is.na(df$sale_price), ]

# bedrooms: mean imputation, rounded to nearest integer
df$bedrooms <- to_num(df$bedrooms)
df$bedrooms[is.na(df$bedrooms)] <- mean(df$bedrooms, na.rm = TRUE)
df$bedrooms <- as.integer(round(df$bedrooms))

# Average and variance of sale price by number of bedrooms
price_by_rooms <- aggregate(sale_price ~ bedrooms, data = df,
                            FUN = function(x) c(avg = mean(x), var = var(x)))

price_by_rooms <- data.frame(
  bedrooms  = price_by_rooms$bedrooms,
  avg_price = round(price_by_rooms$sale_price[, "avg"], 1),
  var_price = round(price_by_rooms$sale_price[, "var"], 1)
)

price_by_rooms


bedrooms,avg_price,var_price
<int>,<dbl>,<dbl>
2,67076.4,565289570
3,154665.1,2378289077
4,234704.6,1725211192
5,301515.9,2484327529
6,375741.3,3924432280


# Task 4

Fit a baseline model to predict the sale price of a house.

 1. Fit your model using the data contained in “train.csv” </br></br>

 2. Use “validation.csv” to predict new values based on your model. You must return a dataframe named `base_result`, that includes `house_id` and `price`. The price column must be your predicted values.

In [10]:
# Use this cell to write your code for Task 4
# ---------- Load data ----------
train      <- read.csv("train.csv", stringsAsFactors = FALSE,
                       na.strings = c("", " ", "NA", "N/A", "NULL"))
validation <- read.csv("validation.csv", stringsAsFactors = FALSE,
                       na.strings = c("", " ", "NA", "N/A", "NULL"))

# ---------- Prepare data ----------
city_levels  <- c("Silvertown", "Riverford", "Teasdale", "Poppleton", "Unknown")
house_levels <- c("Terraced", "Semi-detached", "Detached")

mode_house  <- names(which.max(table(train$house_type)))
mean_months <- mean(train$months_listed, na.rm = TRUE)
mean_beds   <- mean(train$bedrooms, na.rm = TRUE)
mean_area   <- mean(train$area, na.rm = TRUE)

prep <- function(df) {
  df$city[is.na(df$city)] <- "Unknown"
  df$city <- factor(trimws(df$city), levels = city_levels)

  df$house_type[is.na(df$house_type)] <- mode_house
  df$house_type <- factor(trimws(df$house_type), levels = house_levels)

  df$sale_date[is.na(df$sale_date)] <- "2023-01-01"
  df$sale_date <- as.numeric(as.Date(df$sale_date))

  df$months_listed[is.na(df$months_listed)] <- mean_months
  df$bedrooms[is.na(df$bedrooms)]           <- mean_beds
  df$area[is.na(df$area)]                   <- mean_area
  df
}

train      <- prep(train)
validation <- prep(validation)
train      <- train[!is.na(train$sale_price), ]

# ---------- Fit baseline model ----------
base_model <- lm(sale_price ~ city + sale_date + months_listed + bedrooms +
                   house_type + area,
                 data = train)

# ---------- Predict on validation ----------
base_result <- data.frame(
  house_id = validation$house_id,
  price    = predict(base_model, newdata = validation)
)

# ---------- Checks ----------
head(base_result)
sum(is.na(base_result$price))
nrow(base_result) == nrow(validation)

,house_id,price
,<int>,<dbl>
1,1331375,122032.1
2,1630115,303641.2
3,1645745,384158.9
4,1336775,124233.6
5,1888274,271904.8
6,1567679,278011.2


[1] 0

[1] TRUE

# Task 5

Fit a comparison model to predict the sale price of a house.

 1. Fit your model using the data contained in “train.csv” </br></br>

 2. Use “validation.csv” to predict new values based on your model. You must return a dataframe named `compare_result`, that includes `house_id` and `price`. The price column must be your predicted values.

In [11]:
# Use this cell to write your code for Task 5
library(rpart)

# ---------- Load data ----------
train      <- read.csv("train.csv", stringsAsFactors = FALSE,
                       na.strings = c("", " ", "NA", "N/A", "NULL"))
validation <- read.csv("validation.csv", stringsAsFactors = FALSE,
                       na.strings = c("", " ", "NA", "N/A", "NULL"))

# ---------- Prepare data ----------
city_levels  <- c("Silvertown", "Riverford", "Teasdale", "Poppleton", "Unknown")
house_levels <- c("Terraced", "Semi-detached", "Detached")

mode_house  <- names(which.max(table(train$house_type)))
mean_months <- mean(train$months_listed, na.rm = TRUE)
mean_beds   <- mean(train$bedrooms, na.rm = TRUE)
mean_area   <- mean(train$area, na.rm = TRUE)

prep <- function(df) {
  df$city[is.na(df$city)] <- "Unknown"
  df$city <- factor(trimws(df$city), levels = city_levels)

  df$house_type[is.na(df$house_type)] <- mode_house
  df$house_type <- factor(trimws(df$house_type), levels = house_levels)

  df$sale_date[is.na(df$sale_date)] <- "2023-01-01"
  df$sale_date <- as.numeric(as.Date(df$sale_date))

  df$months_listed[is.na(df$months_listed)] <- mean_months
  df$bedrooms[is.na(df$bedrooms)]           <- mean_beds
  df$area[is.na(df$area)]                   <- mean_area
  df
}

train      <- prep(train)
validation <- prep(validation)
train      <- train[!is.na(train$sale_price), ]

# ---------- Fit comparison model (regression tree) ----------
set.seed(42)
compare_model <- rpart(sale_price ~ city + sale_date + months_listed +
                         bedrooms + house_type + area,
                       data    = train,
                       method  = "anova",
                       control = rpart.control(cp = 0.001, minsplit = 10))

# ---------- Predict on validation ----------
compare_result <- data.frame(
  house_id = validation$house_id,
  price    = as.numeric(predict(compare_model, newdata = validation))
)

# ---------- Checks ----------
head(compare_result)
sum(is.na(compare_result$price))
nrow(compare_result) == nrow(validation)

,house_id,price
,<int>,<dbl>
1,1331375,72231.9
2,1630115,330829.3
3,1645745,398565.7
4,1336775,113348.4
5,1888274,285736.9
6,1567679,254798.1


[1] 0

[1] TRUE